# VoxEntity-EU — CPU reproduction notebook (offline)

From the release bundle's frozen predictions: verify hashes and regenerate every table without internet access. This is reproduction level 1 (identical scoring from frozen predictions).

In [ ]:
# 1. verify bundle checksums
import hashlib
from pathlib import Path
bundle = Path('../release_bundle')
bad = []
for line in (bundle / 'checksums.sha256').read_text().splitlines():
    digest, rel = line.split()
    if hashlib.sha256((bundle / rel).read_bytes()).hexdigest() != digest:
        bad.append(rel)
assert not bad, f'checksum mismatch: {bad}'
print('checksums ok')

In [ ]:
# 2. rescore frozen predictions (CPU, deterministic, no network)
from voxentity_eu.cli import main as cli_main
for run in ['whisper-turbo', 'whisper-turbo-slots', 'voxtral-mini', 'voxtral-mini-slots', 'gold-extractor', 'reference-agent']:
    pred = bundle / 'predictions' / f'{run}.jsonl'
    if not pred.exists():
        continue
    rc = cli_main(['score', '--gold', str(bundle / 'gold/scenarios.jsonl'),
                   '--predictions', str(pred),
                   '--manifest', str(bundle / 'audio_manifest.jsonl'),
                   '--out', f'reports/scored_{run}.json'])
    print(run, 'exit', rc)

In [ ]:
# 3. regenerate the report tables
print(cli_main(['report', '--results', 'reports', '--out', 'reports/report.md']))